# 02 — Trees and ridge logistic regression

Fits **CART, OCT, OCT-H and ridge logistic regression** on every feature set (both selectors at k = 3, 5, 10, plus all 41) with 5-fold cross-validation, and saves their outputs in the shared format (`outputs/cv_*.csv`, `oof_*.csv`, `test_*.csv`, `depth_*.csv`). All analysis and plots are in notebook 04.

**Requires:** `outputs/folds.csv` and `outputs/feature_sets.csv` from notebook 01.

In [1]:
include("common.jl")
C = setup()

RUN_MODELS = ["CART", "OCT", "OCT-H", "Logistic"]     # remove a model to skip it (OCT-H is the slowest)
RUN_SETS   = SETS
TREES      = ("CART", "OCT", "OCT-H")
TREE_DEPTHS, TREE_MINBUCKETS = 2:4, [5, 10, 20]
TOL_AUC    = 0.01                                     # simplicity rule
OCTH_SPLIT_SPARSITY = :all                            # max descriptors per OCT-H split (:all or e.g. 2)
LAMBDAS    = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

common.jl loaded: 7 feature sets, outputs in 'outputs', figures in 'figures'.


6-element Vector{Float64}:
   0.001
   0.01
   0.1
   1.0
  10.0
 100.0

---
## Trees: CART, OCT and OCT-H

- **CART** builds the tree greedily, one split at a time.
- **OCT** optimizes all splits of the tree together (IAI local search).
- **OCT-H** is an OCT whose splits may test a weighted sum of descriptors ("0.8·SpMax_L + 0.5·nN ≥ 1.2") instead of one descriptor against a cutoff: slanted boundaries, harder to read.

**Tuning, per fold:** IAI grid search over depth × minimum leaf size on that fold's descriptor list, `cp` tuned by IAI, scored by AUC on the held-out fold. Average over folds; **simplicity rule:** among settings within `TOL_AUC` of the best, take the shallowest, then the largest leaf size. Out-of-fold predictions use that setting; a depth curve records the best result at each depth; the final tree is refit on all 844 chemicals with the final list and the median per-fold `cp`.

In [2]:
tree_kw(kind) = kind == "CART"  ? (localsearch = false,) :
                kind == "OCT-H" ? (hyperplane_config = (sparsity = OCTH_SPLIT_SPARSITY,),) : NamedTuple()
tree_leaves(lnr) = count(n -> IAI.is_leaf(lnr, n), 1:IAI.get_num_nodes(lnr))
splits_of(lnr)   = [n for n in 1:IAI.get_num_nodes(lnr) if !IAI.is_leaf(lnr, n)]
function split_descriptors(lnr, n)
    if IAI.is_hyperplane_split(lnr, n)
        w, _ = IAI.get_split_weights(lnr, n)
        [String(f) for (f, v) in w if v != 0]
    else
        [String(IAI.get_split_feature(lnr, n))]
    end
end
tree_features(lnr) = unique(vcat(String[], [split_descriptors(lnr, n) for n in splits_of(lnr)]...))
max_per_split(lnr) = maximum([length(split_descriptors(lnr, n)) for n in splits_of(lnr)]; init = 0)

function cv_tree(C, s, kind)
    t0 = time()
    per_fold = DataFrame[]
    for f in 1:N_FOLDS
        tr, va, fs = C.fold .!= f, C.fold .== f, C.fold_feats[(s, f)]
        grid = IAI.GridSearch(IAI.OptimalTreeClassifier(; random_seed = SEED, tree_kw(kind)...),
                              max_depth = TREE_DEPTHS, minbucket = TREE_MINBUCKETS)
        IAI.fit!(grid, C.X_dev[tr, fs], C.ys_dev[tr], C.X_dev[va, fs], C.ys_dev[va], validation_criterion = :auc)
        g = select(IAI.get_grid_result_summary(grid), :max_depth, :minbucket, :cp, :valid_score)
        g[!, :fold] .= f
        push!(per_fold, g)
    end
    allf = vcat(per_fold...)
    summ = combine(groupby(allf, [:max_depth, :minbucket]),
                   :valid_score => mean => :cv_auc, :valid_score => std => :cv_sd, :cp => median => :cp)
    ch = sort(summ[summ.cv_auc .>= maximum(summ.cv_auc) - TOL_AUC, :], [:max_depth, order(:minbucket, rev = true)])[1, :]
    fold_cp(f, d, mb) = allf[(allf.fold .== f) .& (allf.max_depth .== d) .& (allf.minbucket .== mb), :cp][1]
    function oof_for(d, mb)
        oof, lv = zeros(length(C.y_dev)), Int[]
        for f in 1:N_FOLDS
            tr, va, fs = C.fold .!= f, C.fold .== f, C.fold_feats[(s, f)]
            lnr = IAI.OptimalTreeClassifier(; random_seed = SEED, tree_kw(kind)..., max_depth = d, minbucket = mb, cp = fold_cp(f, d, mb))
            IAI.fit!(lnr, C.X_dev[tr, fs], C.ys_dev[tr])
            oof[va] = IAI.predict_proba(lnr, C.X_dev[va, fs])[!, "NRB"]
            push!(lv, tree_leaves(lnr))
        end
        oof, lv
    end
    oof, _ = oof_for(ch.max_depth, ch.minbucket)
    dcurve = DataFrame(max_depth = Int[], cv_auc = Float64[], cv_sd = Float64[], cv_bal_acc = Float64[],
                       mean_leaves = Float64[], chosen = Bool[])
    for d in sort(unique(summ.max_depth))
        sdd = summ[summ.max_depth .== d, :]
        b = sdd[argmax(sdd.cv_auc), :]
        od, lv = oof_for(d, b.minbucket)
        push!(dcurve, (d, b.cv_auc, b.cv_sd, metrics(od, C.y_dev; t = best_threshold(od, C.y_dev)).bal_acc, mean(lv), d == ch.max_depth))
    end
    fs = C.final_feats[s]
    final = IAI.OptimalTreeClassifier(; random_seed = SEED, tree_kw(kind)..., max_depth = ch.max_depth, minbucket = ch.minbucket, cp = ch.cp)
    IAI.fit!(final, C.X_dev[:, fs], C.ys_dev)
    (settings = "max_depth=$(ch.max_depth), minbucket=$(ch.minbucket), cp=$(round(ch.cp, sigdigits = 3))",
     cv_auc = ch.cv_auc, cv_sd = ch.cv_sd, oof = oof, test = IAI.predict_proba(final, C.X_te[:, fs])[!, "NRB"],
     model = final, dcurve = dcurve, time = time() - t0)
end

cv_tree (generic function with 1 method)

---
## Ridge logistic regression

Logistic regression with a ridge penalty λ on the coefficients (intercept not penalized), fitted by Newton's method on descriptors standardized on the training part. λ is chosen by mean 5-fold AUC (`cv_generic` in `common.jl`). On a k-descriptor list it is a formula with k weights.

In [3]:
function ridge_logit(Z, y, λ; iters = 100)
    A = hcat(ones(size(Z, 1)), Z)
    P = Diagonal(vcat(0.0, fill(λ, size(Z, 2))))
    b = zeros(size(A, 2))
    for _ in 1:iters
        p = 1 ./ (1 .+ exp.(-A * b))
        step = (A' * (A .* (p .* (1 .- p))) + P) \ (A' * (y .- p) - P * b)
        b .+= step
        norm(step) < 1e-8 && break
    end
    b
end
function fit_logit(X, y, fs, prm)
    mu, sd = standardize_fit(X, fs)
    (b = ridge_logit(zs(X, fs, mu, sd), Float64.(y), prm.lambda), mu = mu, sd = sd, fs = fs)
end
predict_logit(m, X) = 1 ./ (1 .+ exp.(-(hcat(ones(nrow(X)), zs(X, m.fs, m.mu, m.sd)) * m.b)))

predict_logit (generic function with 1 method)

---
## Run and save

One output file set per model. Progress is printed.

In [4]:
final_trees = Dict{Tuple{String,String},Any}()
for m in RUN_MODELS
    st = new_store()
    for s in RUN_SETS
        if m in TREES
            r = cv_tree(C, s, m)
            record!(st, C, m, s, r; leaves = tree_leaves(r.model), used = length(tree_features(r.model)), maxps = max_per_split(r.model))
            final_trees[(s, m)] = r.model
        else
            r = cv_generic(C, s, [(lambda = λ,) for λ in LAMBDAS], fit_logit, predict_logit)
            record!(st, C, m, s, r)
        end
        println(rpad(m, 9), rpad(setlabel(s), 12), "CV AUC ", round(r.cv_auc, digits = 3), " ± ", round(r.cv_sd, digits = 3),
                " | ", r.settings, " | ", round(r.time, digits = 1), " s")
    end
    save_store(st, m)
end

[ Warning: This copy of Interpretable AI software is for academic purposes only and not for commercial use.


CART     k=3 reg.    CV AUC 0.835 ± 0.009 | max_depth=4, minbucket=20, cp=0.00987 | 17.8 s
CART     k=3 log.    CV AUC 0.831 ± 0.043 | max_depth=4, minbucket=20, cp=0.011 | 0.3 s
CART     k=5 reg.    CV AUC 0.824 ± 0.03 | max_depth=4, minbucket=20, cp=0.011 | 0.4 s
CART     k=5 log.    CV AUC 0.815 ± 0.034 | max_depth=4, minbucket=5, cp=0.0121 | 0.4 s
CART     k=10 reg.   CV AUC 0.814 ± 0.025 | max_depth=3, minbucket=10, cp=0.00219 | 0.4 s
CART     k=10 log.   CV AUC 0.832 ± 0.032 | max_depth=4, minbucket=10, cp=0.00219 | 0.4 s
CART     all 41      CV AUC 0.828 ± 0.038 | max_depth=4, minbucket=20, cp=0.0121 | 0.8 s
Saved outputs for CART → outputs/*_cart.csv
OCT      k=3 reg.    CV AUC 0.816 ± 0.012 | max_depth=4, minbucket=20, cp=0.00658 | 17.8 s
OCT      k=3 log.    CV AUC 0.836 ± 0.035 | max_depth=4, minbucket=20, cp=0.011 | 18.1 s
OCT      k=5 reg.    CV AUC 0.843 ± 0.019 | max_depth=4, minbucket=20, cp=0.00731 | 24.8 s
OCT      k=5 log.    CV AUC 0.839 ± 0.018 | max_depth=4, minbu

---
## The final trees

Printed as text and saved as interactive HTML in `figures/trees/`. These are the trees to interpret in the report: the 3-descriptor OCT with each selector and the all-descriptor OCT (plus OCT-H at k = 5).

In [5]:
mkpath(joinpath(FIG_DIR, "trees"))
for key in [("reg_k3", "OCT"), ("log_k3", "OCT"), ("all", "OCT"), ("reg_k5", "OCT-H"), ("log_k5", "OCT-H")]
    haskey(final_trees, key) || continue
    lnr = final_trees[key]
    println("\n=== $(key[2]) | $(setlabel(key[1])) | $(tree_leaves(lnr)) leaves | splits on: ", join(tree_features(lnr), ", "), " ===")
    println(lnr)
    IAI.write_html(joinpath(FIG_DIR, "trees", "tree_$(key[1])_$(model_tag(key[2])).html"), lnr)
end


=== OCT | k=3 reg. | 9 leaves | splits on: LOC, SpPosA_B(p), SpMax_L ===
Fitted OptimalTreeClassifier:
  1) Split: LOC < 1.232
    2) Split: SpPosA_B(p) < 1.264
      3) Split: LOC < 0.546
        4) Predict: NRB (80.00%), [16,4], 20 points, error 0.2
        5) Split: SpMax_L < 4.79
          6) Predict: RB (72.60%), [20,53], 73 points, error 0.274
          7) Predict: NRB (82.65%), [81,17], 98 points, error 0.1735
      8) Predict: NRB (88.24%), [240,32], 272 points, error 0.1176
    9) Split: SpPosA_B(p) < 1.22
      10) Split: SpMax_L < 5.081
        11) Predict: RB (76.09%), [44,140], 184 points, error 0.2391
        12) Predict: NRB (85.71%), [42,7], 49 points, error 0.1429
      13) Split: SpMax_L < 4.721
        14) Split: SpPosA_B(p) < 1.268
          15) Predict: RB (69.70%), [10,23], 33 points, error 0.303
          16) Predict: NRB (95.83%), [23,1], 24 points, error 0.04167
        17) Predict: NRB (91.21%), [83,8], 91 points, error 0.08791

=== OCT | k=3 log. | 6 leaves 

> ### Reading the trees
> For each tree: the path to the main RB leaf and its size, the purest NRB leaf, and the most mixed leaf (where errors concentrate). Read splits as properties: SpMax_L = size/branching, nHM = heavy atoms (e.g. halogens), nN = nitrogen count.
>
> In our earlier run, the 3-descriptor OCT with the **logistic** selector had 6 leaves with whole-number rules: any heavy atom → NRB (91%); two or more nitrogens → NRB (91%); no heavy atom, no nitrogen and smaller (SpMax_L < 4.98) → RB (78%). The regression-selector tree had 9 leaves splitting on LOC and SpPosA_B(p), much harder to explain.